In [ ]:
!pip install -q dspy scikit-learn

## Part 1: Scikit-Learn Classifier:
Darian

This section trains a simple machine learning classifier that will later be exposed as a tool for the DSPy agent.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

Load the Iris dataset:

In [ ]:
iris = load_iris()

X = iris.data
y = iris.target

print("Feature names:", iris.feature_names)
print("Flower types:", iris.target_names)
print("Number of examples:", len(X))

Feature names: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Flower types: ['setosa' 'versicolor' 'virginica']
Number of examples: 150


this is our tiny stand-in for the eventual blight model:
Iris:
flower measurements → ML model → flower type

Eventually:
property features → blight model → blight risk

Split into training and testing data:

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training examples:", len(X_train))
print("Testing examples:", len(X_test))

Training examples: 120
Testing examples: 30


Train the Random Forest

In [ ]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

print("Model trained!")

Model trained!


Test the model:

In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Model accuracy:", accuracy)

Model accuracy: 1.0


Turn model into a function

### Prediction Tool

The `predict_flower` function wraps the trained Random Forest model in a simple Python function. It takes four flower measurements as input and returns the predicted Iris species and the model's confidence.

This function will be provided to the DSPy agent as a tool, and allows the LLM to call the machine learning model when it needs to make a prediction.

In [ ]:
def predict_flower(
    sepal_length: float,
    sepal_width: float,
    petal_length: float,
    petal_width: float
) -> dict:
    """Predict an Iris species from sepal and petal measurements.

    Measurements are in centimeters. Returns the predicted species and
    the Random Forest model's probability for that prediction.
    """
    measurements = [[
        sepal_length,
        sepal_width,
        petal_length,
        petal_width
    ]]

    prediction = model.predict(measurements)[0]
    probabilities = model.predict_proba(measurements)[0]



    flower_name = str(iris.target_names[prediction])

    return {
        "prediction": flower_name,
        "confidence": float(probabilities[prediction])
    }

In [ ]:
predict_flower(5.1, 3.5, 1.4, 0.2)

{'prediction': 'setosa', 'confidence': 1.0}

## Part 2: DSPy Agent & Tool Integration
Nick
This section connects the trained scikit-learn classifier to a DSPy agent. The agent will be able to call the predict_flower function as a tool and use the model's prediction to generate a natural-language response.

In [ ]:
import dspy
from google.colab import userdata

api_key = userdata.get("CapstoneTestKey")

lm = dspy.LM("openai/gpt-5-nano", api_key=api_key)
dspy.configure(lm=lm)

print("DSPy is connected to the language model.")

DSPy is connected to the language model.


This function provides a simple interface to the trained Random Forest. It accepts four flower measurements and returns the predicted species and the model's confidence. DSPy will use the function's name, type hints, and description to understand when to call it.

The signature describes what the agent receives and returns. The ReAct agent can call the `predict_flower` function when a question includes flower measurements, then use the classifier's result to form an answer.

In [ ]:
class FlowerAssistant(dspy.Signature):

    """Answer questions about Iris flower predictions.

    Use the conversation history to understand follow-up questions.
    When the current question gives four flower measurements in centimeters,
    call the predict_flower tool and report its prediction and confidence.
    Use the classifier's result rather than guessing from memory.
    """
    conversation_history: str = dspy.InputField()
    question: str = dspy.InputField()
    answer: str = dspy.OutputField()

flower_agent = dspy.ReAct(
    FlowerAssistant,
    tools=[predict_flower],
    max_iters=3
)

print("Flower assistant with conversation memory is ready.")

Flower assistant with conversation memory is ready.


This test asks the agent to identify a flower from its measurements. The agent should call the Random Forest tool and return the predicted species and confidence in a natural-language response.

In [ ]:
result = flower_agent(
    question="Identify a flower with sepal length 5.1 cm, sepal width 3.5 cm, petal length 1.4 cm, and petal width 0.2 cm."
)

print(result.answer)

2026/10/14 20:07:55 WARNING dspy.predict.predict: Not all input fields were provided to module. Present: ['question', 'trajectory']. Missing: ['conversation_history'].


LMAuthError: [openai/gpt-5-nano] You do not have access to the organization tied to the API key. (invalid_organization)

  credential came from: an explicit api_key (value never shown)

  To fix:
    - Check that your API key is correct and not expired
    - Pass the key explicitly (api_key, or RouterConfig api_keys), or on a host with an environment set OPENAI_API_KEY=...
    - Verify your openai-chat account/project has access


This next cell interrogates the LLM to see that its using the tool, not drawing from its own knowledge.

In [ ]:
for key, value in result.trajectory.items():
    if key.startswith(("tool_name_", "tool_args_", "observation_")):
        print(f"{key}: {value}")

The following cell contains more tests

In [ ]:
test_cases = [
    ("setosa",     (5.1, 3.5, 1.4, 0.2)),
    ("versicolor", (6.0, 2.9, 4.5, 1.5)),
    ("virginica",  (6.5, 3.0, 5.2, 2.0)),
]

for expected, measurements in test_cases:
    sepal_length, sepal_width, petal_length, petal_width = measurements

    result = flower_agent(
        question=(
            f"Identify the flower with sepal length {sepal_length} cm, "
            f"sepal width {sepal_width} cm, petal length {petal_length} cm, "
            f"and petal width {petal_width} cm."
        )
    )

    print(f"\nExample species: {expected}")
    print("Agent answer:", result.answer)

    for key, value in result.trajectory.items():
        if key.startswith(("tool_name_", "observation_")):
            print(f"{key}: {value}")

### Install Gradio

Gradio provides the chat interface where users can enter questions and see the assistant's responses.

In [ ]:
!pip install -q gradio

### Launch the live Iris assistant

Gradio passes the current message and previous messages to the chat function. The chat function formats that history and sends it to the DSPy agent. Chat history is saved in the user's browser so they can revisit conversations there.

In [ ]:
import gradio as gr

def chat(message, history):
    recent_history = history[-8:]  # Keep the last four user/assistant turns
    history_lines = []

    for turn in recent_history:
        role = turn.get("role", "")
        content = turn.get("content", "")
        if isinstance(content, str) and role in ("user", "assistant"):
            history_lines.append(f"{role}: {content}")

    conversation_history = "\n".join(history_lines) or "No earlier conversation."

    result = flower_agent(
        conversation_history=conversation_history,
        question=message
    )
    return result.answer


demo = gr.ChatInterface(
    fn=chat,
    title="Iris Classifier Assistant",
    description=(
        "Ask about Iris flowers or enter sepal and petal measurements "
        "in centimeters to get a classifier prediction."
    ),
    examples=[
        "Identify a flower with sepal length 5.1 cm, sepal width 3.5 cm, "
        "petal length 1.4 cm, and petal width 0.2 cm."
    ],
    save_history=True,
)

demo.launch(share=True)

##Part 3:  Testing & Civic AI Connection
Marissa
This section tests the completed DSPy agent with different inputs to verify that it correctly uses the machine learning tool. It also explains how this prototype could be adapted to the Baton Rouge blight model and used as part of the Civic AI interface.

This function wraps a model that evaluates property features (like structural damage or overgrown lots) to output a blight risk score.

In [ ]:
def predict_blight_risk(
    overgrown_grass: bool,
    structural_damage: bool,
    open_code_violations: int,
    vacant_duration_months: int
) -> dict:
    """Predict the blight risk level for a property based on inspection features.

    Returns risk level ('Low', 'Medium', 'High') and a priority score.
    """
    # Simple rule-based logic representing your ML model score
    score = 0
    if overgrown_grass:
        score += 25
    if structural_damage:
        score += 40
    score += min(open_code_violations * 10, 25)
    score += min(vacant_duration_months * 1, 10)

    if score >= 60:
        risk_level = "High"
    elif score >= 30:
        risk_level = "Medium"
    else:
        risk_level = "Low"

    return {
        "blight_risk": risk_level,
        "priority_score": score,
        "recommended_action": "Schedule immediate inspection" if risk_level == "High" else "Monitor status"
    }

# Test the tool locally
print(predict_blight_risk(overgrown_grass=True, structural_damage=True, open_code_violations=2, vacant_duration_months=12))

Defining a DSPy signature that instructs the ReAct agent to use your new tool when a user provides property assessment data.

In [ ]:
class BlightAssistant(dspy.Signature):
    """Answer questions about property conditions and blight risk in Baton Rouge.

    Use conversation history for context. When property inspection features
    are provided, call the predict_blight_risk tool and report the risk level,
    priority score, and recommended action.
    """
    conversation_history: str = dspy.InputField()
    question: str = dspy.InputField()
    answer: str = dspy.OutputField()

blight_agent = dspy.ReAct(
    BlightAssistant,
    tools=[predict_blight_risk],
    max_iters=3
)

print("Blight risk assistant is ready.")

Testing the agent with various property scenarios to ensure it correctly invokes the custom ML tool instead of guessing.

In [ ]:
blight_test_cases = [
    "Assess a property with overgrown grass: False, structural damage: False, 2 open code violations, vacant for 12 months.",
    "Check a clean property: overgrown grass: False, structural damage: False, 0 violations, vacant for 1 month."
]

for query in blight_test_cases:
    result = blight_agent(conversation_history="No earlier conversation.", question=query)
    print(f"\nQuery: {query}")
    print(f"Agent Answer: {result.answer}")

Updating Gradio app interface to handle civic inquiries, allowing city planners or inspectors to query properties in natural language.

In [ ]:
import gradio as gr

def civic_chat(message, history):
    recent_history = history[-8:]
    history_lines = []

    for turn in recent_history:
        role = turn.get("role", "")
        content = turn.get("content", "")
        if isinstance(content, str) and role in ("user", "assistant"):
            history_lines.append(f"{role}: {content}")

    conversation_history = "\n".join(history_lines) or "No earlier conversation."

    # Switch agent depending on user context or use blight_agent directly
    result = blight_agent(
        conversation_history=conversation_history,
        question=message
    )
    return result.answer

demo_civic = gr.ChatInterface(
    fn=civic_chat,
    title="Baton Rouge Civic AI Blight Assistant",
    description="Ask about property status or enter inspection metrics to calculate blight risk scores.",
    examples=[
        "Assess a property with overgrown grass: True, structural damage: True, 3 open code violations, vacant for 18 months."
    ],
    save_history=True,
)

demo_civic.launch(share=True)